# **Install dependency**

In [0]:
%pip install --quiet pymongo

## **MongoDB extractor**

In [0]:

# ============================================================
# BANGLAPAY | PHASE 1
# MONGODB BRONZE INITIAL SNAPSHOT
#
# Environment : DEV
# Database    : banglapay_ops
# Collections : 6
# Data volume : Deterministic 20%
# Landing     : Unity Catalog Managed Volume
# Format      : Extended JSON Canonical
#
# Source access: READ ONLY
# Snapshot type: INITIAL_SNAPSHOT
# ============================================================

import os
import hashlib
import uuid

from datetime import datetime, timezone

from pymongo import MongoClient, ASCENDING
from bson import json_util
from bson.json_util import CANONICAL_JSON_OPTIONS

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    TimestampType
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

ENVIRONMENT = "dev"

DATABASE_NAME = "banglapay_ops"

SAMPLE_PERCENT = 20
EXPECTED_COLLECTION_COUNT = 6

SECRET_SCOPE = "banglapay-ingestion"

VOLUME_ROOT = (
    "/Volumes/banglapay/ops/batch_landing/mongodb"
)

MANIFEST_TABLE = (
    "banglapay.ops.mongodb_batch_manifest"
)

FETCH_SIZE = 1000

SNAPSHOT_TYPE = "INITIAL_SNAPSHOT"

BATCH_ID = str(uuid.uuid4())

EXTRACTED_AT = datetime.now(timezone.utc)

EXPECTED_COLLECTIONS = [
    "app_sessions",
    "customer_activities",
    "device_profiles",
    "fraud_cases",
    "merchant_device_events",
    "support_cases"
]

assert ENVIRONMENT == "dev"
assert SAMPLE_PERCENT == 20


# ============================================================
# 2. LOAD MONGODB CONNECTION SECRET
# ============================================================

MONGODB_URI = dbutils.secrets.get(
    SECRET_SCOPE,
    "MONGODB_CONNECTION_URI"
).strip()

if not MONGODB_URI:
    raise RuntimeError(
        "MongoDB connection configuration is empty."
    )


# ============================================================
# 3. DETERMINISTIC SAMPLE
#
# Same MongoDB _id always maps to the same sample bucket.
# No random sampling.
# ============================================================

def belongs_to_sample(document_id):

    canonical_id = json_util.dumps(
        document_id,
        json_options=CANONICAL_JSON_OPTIONS
    )

    digest = hashlib.sha256(
        canonical_id.encode("utf-8")
    ).digest()

    bucket = (
        int.from_bytes(digest[:8], "big") % 100
    )

    return bucket < SAMPLE_PERCENT


# ============================================================
# 4. MANIFEST TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MANIFEST_TABLE} (

    batch_id STRING,
    environment STRING,

    source_system STRING,
    source_database STRING,
    source_collection STRING,

    bronze_table STRING,

    snapshot_type STRING,
    sample_percent INT,

    scanned_documents BIGINT,
    extracted_documents BIGINT,

    file_path STRING,

    extracted_at TIMESTAMP

)
USING DELTA

COMMENT 'Append-only MongoDB Bronze batch extraction manifest.'
""")


# ============================================================
# 5. EXTRACT COLLECTIONS
# ============================================================

os.makedirs(
    VOLUME_ROOT,
    exist_ok=True
)

mongo_client = MongoClient(
    MONGODB_URI,

    serverSelectionTimeoutMS=15000,
    connectTimeoutMS=10000,
    socketTimeoutMS=30000,

    appname="BanglaPay-DEV-Bronze-Extractor"
)

manifest_rows = []
collection_results = []

try:

    # Verify Atlas access.
    mongo_client.admin.command("ping")

    mongo_db = mongo_client[DATABASE_NAME]

    collection_names = sorted(
        mongo_db.list_collection_names()
    )

    if collection_names != sorted(
        EXPECTED_COLLECTIONS
    ):
        raise RuntimeError(
            "MongoDB collection inventory changed. "
            f"Expected {EXPECTED_COLLECTIONS}; "
            f"received {collection_names}."
        )

    print("=" * 65)
    print("BANGLAPAY MONGODB BRONZE EXTRACTION")
    print("=" * 65)

    print("Environment:", ENVIRONMENT.upper())
    print("Database:", DATABASE_NAME)
    print("Batch ID:", BATCH_ID)
    print("Data volume:", SAMPLE_PERCENT, "%")
    print("Collections:", len(collection_names))

    for index, collection_name in enumerate(
        collection_names,
        start=1
    ):

        print(
            f"\n[{index}/{EXPECTED_COLLECTION_COUNT}] "
            f"{collection_name}"
        )

        collection = mongo_db[collection_name]

        bronze_table = f"mongo_{collection_name}"

        collection_directory = (
            f"{VOLUME_ROOT}/"
            f"{collection_name}/"
            f"batch_id={BATCH_ID}"
        )

        os.makedirs(
            collection_directory,
            exist_ok=True
        )

        temporary_path = (
            f"{collection_directory}/part-00000.ndjson.tmp"
        )

        final_path = (
            f"{collection_directory}/part-00000.ndjson"
        )

        scanned_count = 0
        extracted_count = 0

        output_file = None

        try:

            # Stream source documents in _id order.
            # No update/delete/insert operations.
            cursor = (
                collection.find({})
                .sort("_id", ASCENDING)
                .batch_size(FETCH_SIZE)
            )

            for document in cursor:

                scanned_count += 1

                source_id = document["_id"]

                if not belongs_to_sample(source_id):
                    continue

                # Open a file only when there is an
                # actual source document to write.
                if output_file is None:

                    output_file = open(
                        temporary_path,
                        "x",
                        encoding="utf-8"
                    )

                envelope = {

                    "_meta": {

                        "batch_id": BATCH_ID,

                        "environment": ENVIRONMENT,

                        "source_system": "mongodb_atlas",

                        "source_database": DATABASE_NAME,

                        "source_collection": collection_name,

                        "source_primary_key": json_util.dumps(
                            source_id,
                            json_options=CANONICAL_JSON_OPTIONS
                        ),

                        "snapshot_type": SNAPSHOT_TYPE,

                        "sample_percent": SAMPLE_PERCENT,

                        "ingested_at": (
                            EXTRACTED_AT.isoformat()
                        )
                    },

                    "record": document
                }

                # BSON-safe Canonical Extended JSON.
                serialized = json_util.dumps(
                    envelope,
                    json_options=CANONICAL_JSON_OPTIONS,
                    ensure_ascii=False,
                    separators=(",", ":")
                )

                output_file.write(
                    serialized + "\n"
                )

                extracted_count += 1

            # Commit completed local file.
            if output_file is not None:

                output_file.flush()
                os.fsync(output_file.fileno())
                output_file.close()

                output_file = None

                os.replace(
                    temporary_path,
                    final_path
                )

                recorded_path = final_path

            else:

                # Zero selected documents:
                # No empty NDJSON file is created.
                recorded_path = None

            print(
                f"Scanned: {scanned_count:,} | "
                f"Extracted: {extracted_count:,}"
            )

        except Exception:

            if output_file is not None:
                output_file.close()

            for path in (
                temporary_path,
                final_path
            ):
                try:
                    os.remove(path)
                except FileNotFoundError:
                    pass

            raise

        manifest_rows.append({

            "batch_id": BATCH_ID,

            "environment": ENVIRONMENT,

            "source_system": "mongodb_atlas",

            "source_database": DATABASE_NAME,

            "source_collection": collection_name,

            "bronze_table": bronze_table,

            "snapshot_type": SNAPSHOT_TYPE,

            "sample_percent": SAMPLE_PERCENT,

            "scanned_documents": scanned_count,

            "extracted_documents": extracted_count,

            "file_path": recorded_path,

            "extracted_at": EXTRACTED_AT
        })

        collection_results.append({

            "collection": collection_name,

            "bronze_table": bronze_table,

            "scanned_documents": scanned_count,

            "extracted_documents": extracted_count,

            "file_path": recorded_path
        })

finally:

    mongo_client.close()


# ============================================================
# 6. PERSIST MANIFEST
# ============================================================

MANIFEST_SCHEMA = StructType([

    StructField("batch_id", StringType(), False),

    StructField("environment", StringType(), False),

    StructField("source_system", StringType(), False),

    StructField("source_database", StringType(), False),

    StructField("source_collection", StringType(), False),

    StructField("bronze_table", StringType(), False),

    StructField("snapshot_type", StringType(), False),

    StructField("sample_percent", IntegerType(), False),

    StructField("scanned_documents", LongType(), False),

    StructField("extracted_documents", LongType(), False),

    StructField("file_path", StringType(), True),

    StructField("extracted_at", TimestampType(), False)

])

manifest_data = [
    (
        row["batch_id"],
        row["environment"],
        row["source_system"],
        row["source_database"],
        row["source_collection"],
        row["bronze_table"],
        row["snapshot_type"],
        row["sample_percent"],
        row["scanned_documents"],
        row["extracted_documents"],
        row["file_path"],
        row["extracted_at"]
    )
    for row in manifest_rows
]

manifest_df = spark.createDataFrame(
    manifest_data,
    schema=MANIFEST_SCHEMA
)

(
    manifest_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(MANIFEST_TABLE)
)


# ============================================================
# 7. FINAL REPORT
# ============================================================

total_scanned = sum(
    row["scanned_documents"]
    for row in collection_results
)

total_extracted = sum(
    row["extracted_documents"]
    for row in collection_results
)

print("\n" + "=" * 65)
print("MONGODB INITIAL SNAPSHOT COMPLETED")
print("=" * 65)

print("Batch ID:", BATCH_ID)
print("Collections extracted:", len(collection_results))
print("Total scanned:", f"{total_scanned:,}")
print("Total extracted:", f"{total_extracted:,}")
print("Data volume:", SAMPLE_PERCENT, "%")
print("Manifest:", MANIFEST_TABLE)

display(
    spark.createDataFrame(collection_results)
    .orderBy("collection")
)

print("SOURCE DATA WAS NOT MODIFIED.")
